# Anomaly Detection with World Models

A world model trained on "normal" sequences learns what's physically plausible.
Anything it hasn't seen — teleportation, clipping through walls, sudden jumps —
gets flagged as anomalous.

This notebook builds a simple anomaly detector from scratch:
1. Generate synthetic "normal" data (a ball bouncing in a box)
2. Generate "anomalous" data (ball teleporting, passing through walls)
3. Train a world model on normal data only
4. Use plausibility scoring to detect anomalies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/03_anomaly_detection.ipynb)

**Time to run:** ~8 minutes on Colab T4

In [ ]:
!pip install "worldkit[train]" -q

## 1. Generate Synthetic Data

We'll create a simple 2D physics simulation: a colored ball bouncing
inside a box. The ball obeys basic kinematics — constant velocity,
elastic wall collisions.

This is our definition of "normal".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import h5py
from pathlib import Path


def render_frame(x, y, size=96, radius=8, ball_color=(70, 130, 230)):
    """Render a ball at position (x, y) on a dark background."""
    frame = np.full((size, size, 3), 30, dtype=np.uint8)  # dark gray background
    # Draw border
    frame[:2, :] = frame[-2:, :] = frame[:, :2] = frame[:, -2:] = [100, 100, 100]
    # Draw ball (filled circle)
    yy, xx = np.ogrid[:size, :size]
    mask = (xx - x) ** 2 + (yy - y) ** 2 <= radius ** 2
    frame[mask] = ball_color
    return frame


def generate_normal_episode(timesteps=32, size=96, radius=8, seed=None):
    """Ball bouncing with elastic wall collisions."""
    rng = np.random.RandomState(seed)
    # Random start position (away from walls)
    x = rng.uniform(radius + 5, size - radius - 5)
    y = rng.uniform(radius + 5, size - radius - 5)
    # Random velocity
    vx = rng.uniform(2, 5) * rng.choice([-1, 1])
    vy = rng.uniform(2, 5) * rng.choice([-1, 1])

    frames = []
    for _ in range(timesteps):
        frames.append(render_frame(int(x), int(y), size, radius))
        x += vx
        y += vy
        # Elastic wall bounce
        if x <= radius or x >= size - radius:
            vx = -vx
            x = np.clip(x, radius, size - radius)
        if y <= radius or y >= size - radius:
            vy = -vy
            y = np.clip(y, radius, size - radius)

    return np.array(frames, dtype=np.uint8)


# Visualize a normal episode
episode = generate_normal_episode(seed=42)
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(episode[i * 4])
    ax.set_title(f"t={i * 4}")
    ax.axis("off")
plt.suptitle("Normal: ball bouncing in a box", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Generate Anomalous Data

Now we create sequences that violate the physics:
- **Teleportation**: ball jumps to a random position mid-sequence
- **Wall clipping**: ball passes through a wall instead of bouncing
- **Speed anomaly**: ball suddenly accelerates or stops

In [ ]:
def generate_teleport_episode(timesteps=32, size=96, radius=8, seed=None):
    """Ball bounces normally, then teleports to a random location."""
    rng = np.random.RandomState(seed)
    x = rng.uniform(radius + 5, size - radius - 5)
    y = rng.uniform(radius + 5, size - radius - 5)
    vx = rng.uniform(2, 5) * rng.choice([-1, 1])
    vy = rng.uniform(2, 5) * rng.choice([-1, 1])
    teleport_at = rng.randint(timesteps // 3, 2 * timesteps // 3)

    frames = []
    for t in range(timesteps):
        if t == teleport_at:
            # Teleport to random position
            x = rng.uniform(radius + 5, size - radius - 5)
            y = rng.uniform(radius + 5, size - radius - 5)
        frames.append(render_frame(int(x), int(y), size, radius))
        x += vx
        y += vy
        if x <= radius or x >= size - radius:
            vx = -vx
            x = np.clip(x, radius, size - radius)
        if y <= radius or y >= size - radius:
            vy = -vy
            y = np.clip(y, radius, size - radius)

    return np.array(frames, dtype=np.uint8)


def generate_freeze_episode(timesteps=32, size=96, radius=8, seed=None):
    """Ball bounces normally, then freezes in place."""
    rng = np.random.RandomState(seed)
    x = rng.uniform(radius + 5, size - radius - 5)
    y = rng.uniform(radius + 5, size - radius - 5)
    vx = rng.uniform(2, 5) * rng.choice([-1, 1])
    vy = rng.uniform(2, 5) * rng.choice([-1, 1])
    freeze_at = rng.randint(timesteps // 3, 2 * timesteps // 3)

    frames = []
    for t in range(timesteps):
        frames.append(render_frame(int(x), int(y), size, radius))
        if t < freeze_at:
            x += vx
            y += vy
            if x <= radius or x >= size - radius:
                vx = -vx
                x = np.clip(x, radius, size - radius)
            if y <= radius or y >= size - radius:
                vy = -vy
                y = np.clip(y, radius, size - radius)
        # After freeze_at, ball stays still

    return np.array(frames, dtype=np.uint8)


def generate_wallclip_episode(timesteps=32, size=96, radius=8, seed=None):
    """Ball passes through walls instead of bouncing."""
    rng = np.random.RandomState(seed)
    x = rng.uniform(radius + 5, size - radius - 5)
    y = rng.uniform(radius + 5, size - radius - 5)
    vx = rng.uniform(2, 5) * rng.choice([-1, 1])
    vy = rng.uniform(2, 5) * rng.choice([-1, 1])

    frames = []
    for _ in range(timesteps):
        # Wrap position instead of bouncing (passes through walls)
        draw_x = int(x) % size
        draw_y = int(y) % size
        frames.append(render_frame(draw_x, draw_y, size, radius))
        x += vx
        y += vy

    return np.array(frames, dtype=np.uint8)


# Visualize each anomaly type
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
labels = ["Teleportation", "Freeze", "Wall clipping"]
generators = [generate_teleport_episode, generate_freeze_episode, generate_wallclip_episode]

for row, (label, gen) in enumerate(zip(labels, generators)):
    ep = gen(seed=42)
    for col in range(8):
        axes[row, col].imshow(ep[col * 4])
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(label, fontsize=12, rotation=0, labelpad=80)

plt.suptitle("Anomalous sequences", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build the Training Dataset

We train the model on **normal data only**. The model learns what
physically plausible motion looks like. Anything outside this
distribution gets a low plausibility score.

In [ ]:
# Generate normal training data: 300 episodes
n_episodes = 300
timesteps = 32

all_pixels = []
all_actions = []

for i in range(n_episodes):
    ep = generate_normal_episode(timesteps=timesteps, seed=i)
    all_pixels.append(ep)
    # Dummy actions (ball moves on its own, but we need the field)
    all_actions.append(np.zeros((timesteps, 2), dtype=np.float32))

pixels = np.array(all_pixels, dtype=np.uint8)  # (300, 32, 96, 96, 3)
actions = np.array(all_actions, dtype=np.float32)  # (300, 32, 2)

with h5py.File("bouncing_ball_normal.h5", "w") as f:
    f.create_dataset("pixels", data=pixels, compression="gzip")
    f.create_dataset("actions", data=actions, compression="gzip")

print(f"Training data: {pixels.shape[0]} episodes, {pixels.shape[1]} timesteps")
print(f"Frame size: {pixels.shape[2]}x{pixels.shape[3]}")

## 4. Train the World Model

Train a `nano` model on the normal bouncing ball data.
After training, the model should understand that balls move smoothly
and bounce off walls.

In [ ]:
from worldkit import WorldModel

model = WorldModel.train(
    data="bouncing_ball_normal.h5",
    config="nano",
    epochs=30,
    batch_size=32,
    action_dim=2,
    device="auto",
)

print(f"\nTrained: {model.num_params:,} parameters")

## 5. Score Normal vs Anomalous Sequences

Now the key test: does the model assign higher plausibility scores
to normal sequences and lower scores to anomalous ones?

In [ ]:
n_test = 50

# Score normal sequences (using seeds the model hasn't seen)
normal_scores = []
for i in range(n_test):
    ep = generate_normal_episode(timesteps=16, seed=1000 + i)
    score = model.plausibility(list(ep))
    normal_scores.append(score)

# Score teleportation anomalies
teleport_scores = []
for i in range(n_test):
    ep = generate_teleport_episode(timesteps=16, seed=2000 + i)
    score = model.plausibility(list(ep))
    teleport_scores.append(score)

# Score freeze anomalies
freeze_scores = []
for i in range(n_test):
    ep = generate_freeze_episode(timesteps=16, seed=3000 + i)
    score = model.plausibility(list(ep))
    freeze_scores.append(score)

# Score wall-clipping anomalies
wallclip_scores = []
for i in range(n_test):
    ep = generate_wallclip_episode(timesteps=16, seed=4000 + i)
    score = model.plausibility(list(ep))
    wallclip_scores.append(score)

print(f"Normal mean:       {np.mean(normal_scores):.4f} +/- {np.std(normal_scores):.4f}")
print(f"Teleport mean:     {np.mean(teleport_scores):.4f} +/- {np.std(teleport_scores):.4f}")
print(f"Freeze mean:       {np.mean(freeze_scores):.4f} +/- {np.std(freeze_scores):.4f}")
print(f"Wall-clip mean:    {np.mean(wallclip_scores):.4f} +/- {np.std(wallclip_scores):.4f}")

In [ ]:
# Visualize score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
data = [normal_scores, teleport_scores, freeze_scores, wallclip_scores]
labels = ["Normal", "Teleport", "Freeze", "Wall-clip"]
colors = ["steelblue", "coral", "orange", "mediumpurple"]

bp = axes[0].boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel("Plausibility score")
axes[0].set_title("Score distribution by sequence type")

# Histogram
axes[1].hist(normal_scores, bins=15, alpha=0.7, color="steelblue", label="Normal")
axes[1].hist(teleport_scores, bins=15, alpha=0.7, color="coral", label="Teleport")
axes[1].hist(freeze_scores, bins=15, alpha=0.7, color="orange", label="Freeze")
axes[1].hist(wallclip_scores, bins=15, alpha=0.7, color="mediumpurple", label="Wall-clip")
axes[1].set_xlabel("Plausibility score")
axes[1].set_ylabel("Count")
axes[1].set_title("Score histograms")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Build a Simple Anomaly Detector

Set a threshold on the plausibility score.
Anything below the threshold is flagged as anomalous.

In [ ]:
# Use the 5th percentile of normal scores as threshold
threshold = np.percentile(normal_scores, 5)
print(f"Threshold (5th percentile of normal): {threshold:.4f}")

# Evaluate detection performance
all_anomalous = teleport_scores + freeze_scores + wallclip_scores

# True positive rate: fraction of anomalies detected
tp_rate = np.mean([s < threshold for s in all_anomalous])
# False positive rate: fraction of normal sequences flagged
fp_rate = np.mean([s < threshold for s in normal_scores])

print(f"\nDetection rate (anomalies caught): {tp_rate:.1%}")
print(f"False alarm rate (normal flagged):  {fp_rate:.1%}")

# Per-anomaly breakdown
for name, scores in [("Teleport", teleport_scores), ("Freeze", freeze_scores),
                      ("Wall-clip", wallclip_scores)]:
    detected = np.mean([s < threshold for s in scores])
    print(f"  {name}: {detected:.1%} detected")

In [ ]:
# ROC-style curve: detection rate vs false alarm rate at different thresholds
thresholds = np.linspace(0, 1, 200)
tp_rates = []
fp_rates = []

for t in thresholds:
    tp = np.mean([s < t for s in all_anomalous])
    fp = np.mean([s < t for s in normal_scores])
    tp_rates.append(tp)
    fp_rates.append(fp)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fp_rates, tp_rates, color="steelblue", linewidth=2)
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.5, label="Random detector")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (detection rate)")
ax.set_title("Anomaly Detection ROC Curve")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 7. Visualize Anomaly Detection in Action

Let's show a frame-by-frame breakdown: score each sub-sequence
within a longer episode to pinpoint where the anomaly occurs.

In [ ]:
# Generate a long anomalous episode
ep_anomalous = generate_teleport_episode(timesteps=32, seed=5555)

# Score sliding windows of length 6
window_size = 6
window_scores = []
for start in range(len(ep_anomalous) - window_size + 1):
    window = list(ep_anomalous[start:start + window_size])
    score = model.plausibility(window)
    window_scores.append(score)

# Plot frames and scores
fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={"height_ratios": [1, 1.5]})

# Top: show frames
n_show = min(16, len(ep_anomalous))
frame_strip = np.concatenate([ep_anomalous[i] for i in range(n_show)], axis=1)
axes[0].imshow(frame_strip)
axes[0].set_title("Teleportation episode (frame strip)")
axes[0].axis("off")

# Bottom: plot window scores
axes[1].plot(range(len(window_scores)), window_scores, "o-", color="steelblue", markersize=4)
axes[1].axhline(y=threshold, color="coral", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.3f})")
axes[1].fill_between(range(len(window_scores)), 0, threshold, alpha=0.1, color="coral")
axes[1].set_xlabel("Window start frame")
axes[1].set_ylabel("Plausibility score")
axes[1].set_title("Sliding window plausibility")
axes[1].legend()

plt.tight_layout()
plt.show()

# Find the anomaly
min_idx = np.argmin(window_scores)
print(f"Lowest score at window starting frame {min_idx}: {window_scores[min_idx]:.4f}")
print(f"This is where the teleportation likely occurred.")

## Next Steps

This pattern works for any domain where you can define "normal":
- **Manufacturing**: train on normal assembly videos, detect defects
- **Robotics**: train on nominal operation, detect mechanical failures
- **Surveillance**: train on normal activity, detect unusual events
- **Simulation**: train on valid physics, detect simulation bugs

Key ideas:
- You only need labeled "normal" data — no anomaly labels required
- The plausibility score is a continuous confidence measure
- Sliding windows help localize anomalies in time

| Notebook | What's next |
|----------|-------------|
| [04 — Latent Probing](04_latent_probing.ipynb) | Understand what your model learned |
| [05 — Model Comparison](05_model_comparison.ipynb) | Would a larger model detect more anomalies? |
| [06 — Export & Deploy](06_export_deploy.ipynb) | Deploy your anomaly detector to production |